## 1. Importing Dependencies
We start by importing the necessary libraries to create our LangChain application, including `requests` for making API calls and decorators for creating custom tools.


In [97]:
import requests
from langchain.tools import tool
from langchain_core.tools import InjectedToolArg
from typing import Annotated

## 2. Defining Custom LangChain Tools
We define two custom tools using the `@tool` decorator:
- `get_conversion_factor`: Fetches the live exchange rate between two currencies using the ExchangeRate-API.
- `convert`: Multiplies the base currency value by the retrieved conversion rate. Notice how `conversion_rate` uses `InjectedToolArg` so it can be passed dynamically during tool execution without the LLM needing to supply it directly.


In [98]:
# TOOL CREATION
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """this function returns the currency conversion factor between the given base currency and target currrency"""
    url = f"https://v6.exchangerate-api.com/v6/1d92e86028dd874f758df6ab/pair/{base_currency}/{target_currency}"
    response = requests.get(url)

    return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
     """this function returns the target currency  by calculating the conversion factor with the given base currency"""
     return base_currency_value * conversion_rate


### Testing the Tool Manually
We can test our new `get_conversion_factor` tool by directly invoking it with a dictionary of arguments.


In [60]:
conv_rate = get_conversion_factor.invoke({'base_currency': 'USD','target_currency': 'INR'})

In [61]:
conv_rate

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1780531201,
 'time_last_update_utc': 'Thu, 04 Jun 2026 00:00:01 +0000',
 'time_next_update_unix': 1780617601,
 'time_next_update_utc': 'Fri, 05 Jun 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.7911}

We also test the `convert` tool manually using the rate we just fetched.


In [63]:
convert.invoke({'base_currency_value': 19, 'conversion_rate': conv_rate['conversion_rate']})

1820.0309

## 3. Initializing the LLM and Binding Tools
Here we load our HuggingFace API token and initialize the `Qwen/Qwen2.5-72B-Instruct` model using `HuggingFaceEndpoint`.


In [64]:
# TOOL BINDING

from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from dotenv import load_dotenv
import os

In [65]:
load_dotenv()

hf_token = os.getenv("HF_TOKEN")

In [66]:
llm=HuggingFaceEndpoint(
    repo_id='Qwen/Qwen2.5-72B-Instruct',
    task='text-generation',
    huggingfacehub_api_token=hf_token
)

model = ChatHuggingFace(llm=llm)

### Binding Tools
We bind our custom tools to the model so that the LLM is aware of them and can request to use them when necessary.


In [67]:
model_with_tools = model.bind_tools([get_conversion_factor, convert])

In [99]:
from langchain_core.messages import HumanMessage

In [100]:
messages=[]

## 4. Prompting the LLM
We create a conversation list `messages` and add a `HumanMessage` asking the model to perform a currency conversion.


In [101]:
messages = [HumanMessage('What is the conversion factor between USD and INR, and based on that can you convert 10 inr to usd')]

In [102]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={})]

### First Model Invocation
We invoke the model. Instead of returning a direct text response, the model will return an `AIMessage` containing `tool_calls` because it realizes it needs to look up the exchange rate first.


In [103]:
ai_msgs = model_with_tools.invoke(messages)

In [104]:
ai_msgs

AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency": "USD", "target_currency": "INR"}', 'name': 'get_conversion_factor', 'description': None}, 'id': 'call_fx5gH6yIGqD8TVCnXNH2Eico', 'type': 'function'}, {'function': {'arguments': '{"base_currency_value": 10}', 'name': 'convert', 'description': None}, 'id': 'call_hnjDUAoPIstLpWpUQWnvCYst', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 445, 'total_tokens': 497}, 'model_name': 'Qwen/Qwen2.5-72B-Instruct', 'system_fingerprint': '', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e9334-0a11-7030-ad98-3d62c3c73d5d-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_fx5gH6yIGqD8TVCnXNH2Eico', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_currency_value': 10}, 'id': 'call_hnjDUAoPIstLpWpUQWnvCYst', 'type': 'tool_call'}], invalid_tool_calls=[], us

In [105]:
messages.append(ai_msgs)

In [106]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency": "USD", "target_currency": "INR"}', 'name': 'get_conversion_factor', 'description': None}, 'id': 'call_fx5gH6yIGqD8TVCnXNH2Eico', 'type': 'function'}, {'function': {'arguments': '{"base_currency_value": 10}', 'name': 'convert', 'description': None}, 'id': 'call_hnjDUAoPIstLpWpUQWnvCYst', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 445, 'total_tokens': 497}, 'model_name': 'Qwen/Qwen2.5-72B-Instruct', 'system_fingerprint': '', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e9334-0a11-7030-ad98-3d62c3c73d5d-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_fx5gH6yIGqD8TVCnXNH2Eic

In [107]:
ai_msgs.tool_calls[0]

{'name': 'get_conversion_factor',
 'args': {'base_currency': 'USD', 'target_currency': 'INR'},
 'id': 'call_fx5gH6yIGqD8TVCnXNH2Eico',
 'type': 'tool_call'}

In [108]:
tool_msg1 = get_conversion_factor.invoke(ai_msgs.tool_calls[0])
tool_msg1

ToolMessage(content='{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1780531201, "time_last_update_utc": "Thu, 04 Jun 2026 00:00:01 +0000", "time_next_update_unix": 1780617601, "time_next_update_utc": "Fri, 05 Jun 2026 00:00:01 +0000", "base_code": "USD", "target_code": "INR", "conversion_rate": 95.7911}', name='get_conversion_factor', tool_call_id='call_fx5gH6yIGqD8TVCnXNH2Eico')

In [110]:
tool_msg1.content

'{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1780531201, "time_last_update_utc": "Thu, 04 Jun 2026 00:00:01 +0000", "time_next_update_unix": 1780617601, "time_next_update_utc": "Fri, 05 Jun 2026 00:00:01 +0000", "base_code": "USD", "target_code": "INR", "conversion_rate": 95.7911}'

In [111]:
import json

## 5. Executing Tool Calls Programmatically
We loop through the tool calls requested by the model. 
- If it asks for the conversion factor, we invoke the tool and extract the rate.
- If it asks to convert the currency, we inject the rate we got and invoke the convert tool.

Finally, we append the outputs of these tools (as `ToolMessage`s) back to our `messages` history.


In [112]:

for tool_call in ai_msgs.tool_calls:
    if tool_call['name'] == 'get_conversion_factor':
        tool_msg1 = get_conversion_factor.invoke(tool_call)

        data = json.loads(tool_msg1.content)
        conversion_rate=data["conversion_rate"]

        messages.append(tool_msg1)

    if tool_call['name'] == 'convert':
        tool_call['args']['conversion_rate'] = conversion_rate
        tool_msg2 = convert.invoke(tool_call)
        messages.append(tool_msg2)

In [113]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency": "USD", "target_currency": "INR"}', 'name': 'get_conversion_factor', 'description': None}, 'id': 'call_fx5gH6yIGqD8TVCnXNH2Eico', 'type': 'function'}, {'function': {'arguments': '{"base_currency_value": 10}', 'name': 'convert', 'description': None}, 'id': 'call_hnjDUAoPIstLpWpUQWnvCYst', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 445, 'total_tokens': 497}, 'model_name': 'Qwen/Qwen2.5-72B-Instruct', 'system_fingerprint': '', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e9334-0a11-7030-ad98-3d62c3c73d5d-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_fx5gH6yIGqD8TVCnXNH2Eic

## 6. Final Model Invocation
Now that the `messages` list contains the user prompt, the LLM's tool call requests, and the actual tool outputs, we invoke the model one last time. It will use the tool outputs to generate a final natural language response.


In [114]:
res = model_with_tools.invoke(messages)

In [115]:
res

AIMessage(content='The conversion factor between USD and INR is 95.7911. Based on this, 10 INR is approximately 0.1044 USD.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 790, 'total_tokens': 828}, 'model_name': 'Qwen/Qwen2.5-72B-Instruct', 'system_fingerprint': '', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e9334-abf0-7701-9997-dfdf6a57fe21-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 790, 'output_tokens': 38, 'total_tokens': 828})